# Model Context Protocol (MCP) Servers for Science

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

If "tool use" is the *grammar* of agents, **MCP is the standard library**.

The Model Context Protocol (released by Anthropic in late 2024, now an open
standard adopted by Claude, ChatGPT, Cursor, Copilot, etc.) lets you write a
**single tool server once** and have *any* MCP-aware agent connect to it. No
more rewriting the same image-analysis tool wrapper for every host application.

The analogy people use is **USB-C for AI agents**: one connector, many devices.

## What you'll build

We'll take the *exact same* particle-analysis tools you wrote from scratch in
notebook 04 and expose them via MCP — so any MCP-aware host (Claude Desktop,
Claude Code, Cursor, etc.) can use them without us writing host-specific glue.

1. A tiny MCP server exposing four image-processing tools:
   - `inspect_image()` — basic statistics on the loaded image
   - `threshold_image(method)` — Otsu (or fixed) binarization
   - `segment_particles(min_area)` — connected-component labelling
   - `measure_particles()` — radius statistics for the labelled particles
2. A direct client connection — call the tools manually, see the wire format.
3. An **agent** that asks Claude to solve a particle-analysis problem by
   autonomously calling tools through the MCP server.

By the end you'll know enough to wrap any of your group's analysis scripts
into an MCP server and have Claude (or Cursor, or Claude Code) drive them.

---
## Setup

In [ ]:
# !pip install mcp anthropic numpy scipy

In [ ]:
import os, getpass, asyncio, json, subprocess, sys, tempfile, textwrap
import numpy as np

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6" 

---
## Step 1: Write the MCP server

In production you'd put this in `microscopy_mcp_server.py` and ship it with
your analysis package. For the tutorial we write it to a temp file from
inside the notebook so everything stays self-contained.

Key things to notice in the server code:
- `FastMCP("microscopy-tools")` declares the server and its name.
- Each `@mcp.tool()` decorator turns a normal Python function into an MCP tool.
  The function's **type hints** become the JSON schema; the **docstring**
  becomes the description shown to the LLM.
- `mcp.run()` with no arguments uses **stdio transport** — the server reads
  JSON-RPC from stdin and writes responses to stdout, perfect for being spawned
  as a subprocess by an agent.

The server holds the image in module-level state (the same `SESSION` pattern
as notebook 04). For a real deployment you'd accept a file path or an
**MCP resource** identifier instead.

In [ ]:
SERVER_CODE = textwrap.dedent("""
    from mcp.server.fastmcp import FastMCP
    import numpy as np
    from scipy import ndimage

    mcp = FastMCP("microscopy-tools")

    # --- Synthesise a deterministic image so client and server agree ---
    def _make_image(seed=0, n=45, size=512, mean_r=14, std_r=4, gap=4):
        rng = np.random.default_rng(seed)
        img = np.zeros((size, size), dtype=np.float32)
        yy, xx = np.mgrid[:size, :size]
        placed = []
        while len(placed) < n:
            r = float(np.clip(rng.normal(mean_r, std_r), 6, 28))
            cx, cy = rng.uniform(r+2, size-r-2, size=2)
            if any((cx-px)**2 + (cy-py)**2 < (r+pr+gap)**2 for px,py,pr in placed):
                continue
            placed.append((cx, cy, r))
        for cx, cy, r in placed:
            mask = (xx-cx)**2 + (yy-cy)**2 <= r**2
            img[mask] = 0.9
        img += 0.04 * rng.standard_normal(img.shape)
        return np.clip(img, 0, 1)

    SESSION = {"image": _make_image(), "binary": None, "labels": None, "n": 0}

    @mcp.tool()
    def inspect_image() -> dict:
        \"\"\"Return shape and intensity statistics for the loaded image.\"\"\"
        img = SESSION["image"]
        return {"shape": list(img.shape),
                "intensity_mean": float(img.mean()),
                "intensity_std":  float(img.std()),
                "intensity_min":  float(img.min()),
                "intensity_max":  float(img.max())}

    @mcp.tool()
    def threshold_image(method: str = "otsu", value: float | None = None) -> dict:
        \"\"\"Binarize the image into foreground/background.

        Args:
            method: 'otsu' for automatic, or 'fixed' with a value in [0,1].
            value:  Threshold to use when method='fixed'.
        \"\"\"
        img = SESSION["image"]
        if method == "otsu":
            hist, edges = np.histogram(img, bins=256, range=(0, 1))
            p = hist / hist.sum()
            omega = np.cumsum(p)
            mu = np.cumsum(p * (edges[:-1] + 0.5*(edges[1]-edges[0])))
            sigma_b = (mu[-1]*omega - mu)**2 / (omega*(1-omega) + 1e-12)
            thr = float(edges[:-1][np.argmax(sigma_b)])
        elif method == "fixed":
            if value is None:
                return {"error": "method='fixed' requires a value (0-1)"}
            thr = float(value)
        else:
            return {"error": f"unknown method {method!r}"}
        SESSION["binary"] = (img > thr).astype(np.uint8)
        return {"threshold": thr,
                "foreground_fraction": float(SESSION["binary"].mean())}

    @mcp.tool()
    def segment_particles(min_area: int = 30) -> dict:
        \"\"\"Connected-component labelling on the binarized image.\"\"\"
        if SESSION["binary"] is None:
            return {"error": "Call threshold_image first."}
        labels, n = ndimage.label(SESSION["binary"])
        sizes = ndimage.sum(SESSION["binary"], labels, range(1, n+1))
        keep = np.where(sizes >= min_area)[0] + 1
        relabel = np.zeros(n+1, dtype=int)
        relabel[keep] = np.arange(1, len(keep)+1)
        SESSION["labels"], SESSION["n"] = relabel[labels], len(keep)
        return {"n_particles": int(SESSION["n"]), "min_area_used": int(min_area)}

    @mcp.tool()
    def measure_particles() -> dict:
        \"\"\"Mean/median/std equivalent radius (px) over labelled particles.\"\"\"
        labels, n = SESSION["labels"], SESSION["n"]
        if labels is None or n == 0:
            return {"error": "Call segment_particles first."}
        areas = ndimage.sum(np.ones_like(labels), labels, range(1, n+1))
        radii = np.sqrt(areas / np.pi)
        return {"n_particles": int(n),
                "mean_radius_px":   float(radii.mean()),
                "median_radius_px": float(np.median(radii)),
                "std_radius_px":    float(radii.std()),
                "min_radius_px":    float(radii.min()),
                "max_radius_px":    float(radii.max())}

    if __name__ == "__main__":
        mcp.run()
""").strip()

# Write the server to a temp file
SERVER_PATH = os.path.join(tempfile.gettempdir(), "microscopy_mcp_server.py")
with open(SERVER_PATH, "w") as f:
    f.write(SERVER_CODE)

print("Server written to", SERVER_PATH)
print("\n--- first 30 lines ---")
print("\n".join(SERVER_CODE.splitlines()[:30]))

---
## Step 2: Connect a client

The MCP client spawns the server as a subprocess, then talks JSON-RPC over
its stdio. Notice how clean the API is — `list_tools()` and `call_tool()` are
all you need to know.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import subprocess

# Jupyter / Colab gotcha: ipykernel replaces sys.stderr with a stream that has
# no OS file descriptor, which breaks subprocess spawning inside MCP. We
# explicitly hand the spawned server the *real* stderr (or /dev/null).
ERRLOG = sys.__stderr__ if sys.__stderr__ is not None else subprocess.DEVNULL

async def list_and_call_demo():
    params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
    async with stdio_client(params, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 1. Enumerate tools
            tools = await session.list_tools()
            for t in tools.tools:
                print(f"- {t.name}: {t.description.splitlines()[0]}")

            # 2. Call inspect_image directly
            result = await session.call_tool("inspect_image", {})
            print("\ninspect_image() ->")
            print(result.content[0].text)

await list_and_call_demo()

**What just happened:**
1. We launched `python microscopy_mcp_server.py` as a child process.
2. The client opened a JSON-RPC channel over its stdin/stdout.
3. `list_tools()` returned the schemas auto-generated from our Python signatures.
4. `call_tool(...)` ran the function in the *server's* process and returned the result.

Same wire format that Claude Desktop, Claude Code, Cursor, and any other
MCP-aware host uses. *Write the tool once, run it everywhere.*

---
## Step 3: Hand the tools to an agent

Now the punchline: instead of *us* deciding which tool to call, we let Claude
decide. The MCP tool list translates one-to-one into Anthropic's tool-use
format.

In [ ]:
def mcp_tools_to_anthropic(mcp_tools):
    """Convert an MCP tool list into Anthropic's `tools=` parameter format."""
    return [{
        "name":         t.name,
        "description":  t.description or "",
        "input_schema": t.inputSchema,
    } for t in mcp_tools]


async def run_agent(user_question: str, max_turns: int = 8):
    params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
    async with stdio_client(params, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_listing = await session.list_tools()
            anthropic_tools = mcp_tools_to_anthropic(tools_listing.tools)

            messages = [{"role": "user", "content": user_question}]

            for turn in range(max_turns):
                resp = client.messages.create(
                    model=MODEL, max_tokens=1024,
                    tools=anthropic_tools,
                    messages=messages,
                )
                # Append assistant turn (may contain text + tool_use)
                messages.append({"role": "assistant", "content": resp.content})

                tool_uses = [b for b in resp.content if b.type == "tool_use"]
                if not tool_uses:
                    return "".join(b.text for b in resp.content if b.type == "text")

                # Dispatch each tool call through MCP
                tool_results = []
                for tu in tool_uses:
                    print(f"  -> calling {tu.name}({json.dumps(tu.input)[:80]})")
                    out = await session.call_tool(tu.name, tu.input)
                    payload = out.content[0].text
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tu.id,
                        "content": payload,
                    })
                messages.append({"role": "user", "content": tool_results})
            return "[max turns reached]"


answer = await run_agent(
    "A microscopy image is already loaded on the server. Find out how many "
    "particles are in it and what their mean equivalent radius is. Use the "
    "tools rather than guessing, and return a short quantitative summary."
)
print("\n=== Final answer ===\n")
print(answer)

**Trace what happened:**
- Claude saw the question + the four tools.
- It chained `inspect_image` → `threshold_image('otsu')` → `segment_particles`
  → `measure_particles`, then summarised.
- It synthesised a final answer with the actual numbers, not guesses.

You did not write a planner, a router, or a state machine. The model planned
the tool calls itself; MCP gave it the tools to call.

---
## Step 4: A harder question (forces multi-step reasoning)

Same agent, but now the question requires the agent to *try two
configurations and compare* before answering. With MCP, this happens
identically to how you'd do it locally — the protocol is invisible.

In [ ]:
answer = await run_agent(
    "I'm worried small noise blobs are inflating the particle count. "
    "Try segmentation with two different min_area values (e.g. 5 and 200), "
    "compare the resulting counts, recommend a sensible value, and then "
    "report the final count and mean radius at that setting."
)
print("\n=== Final answer ===\n")
print(answer)

---
## Why MCP is a big deal for scientific tooling

| Without MCP | With MCP |
|---|---|
| Each agent host reinvents tool wrappers | Write the tool *once*, expose to all hosts |
| Tool descriptions drift between Claude/GPT/Cursor | Single source of truth (the server) |
| No standard for tool versioning, auth, resources | All in the spec |
| Sharing tools means shipping code | Sharing tools means shipping a server URL |

For our community specifically, this means:
- One person writes a `pymatgen` MCP server → the entire group can use it
  from Claude Desktop, Claude Code, ChatGPT, Cursor.
- Beamline control APIs become *callable from any agent* without per-agent glue.
- Existing analysis scripts (segmentation, peak fitting, Rietveld, fluorescence
  quantification) become first-class agent tools with a 10-line wrapper each.

## Where to go next

- **MCP resources** — expose datasets & files (not just functions).
- **MCP prompts** — share canned prompt templates between hosts.
- **Remote transport** — replace stdio with HTTP/SSE for hosted servers.
- **Authentication** — OAuth, signed requests for tools that touch instruments.

The next notebook moves from *one* agent calling *one* tool to *several*
specialist agents coordinated by a planner — same primitives, larger system.